In [1]:
import scanpy as sc
import pandas as pd
import numpy as np


In [4]:
adata = sc.read_h5ad('../data/mtDNA_DSB_5k_clustered_manual_annotation.h5ad')

/Users/christoffer/miniconda3/envs/sc/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [12]:
adata_raw = sc.read_h5ad('../data/mtDNA_DSB_5k_raw.h5ad')

/Users/christoffer/miniconda3/envs/sc/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [15]:
import numpy as np
import pandas as pd
from scipy import sparse

def _report_dups(name, idx):
    dup = pd.Index(idx)
    d = dup[dup.duplicated()].unique().tolist()
    if d:
        print(f"[WARN] {name}: {len(d)} duplicated names e.g. {d[:5]}")
    else:
        print(f"[OK] {name}: all unique")

def attach_counts_intersection_safe(adata, adata_raw, layer_name="counts", collapse_var=False):
    # 1) report duplicates
    _report_dups("adata.obs_names", adata.obs_names)
    _report_dups("adata.var_names", adata.var_names)
    _report_dups("adata_raw.obs_names", adata_raw.obs_names)
    _report_dups("adata_raw.var_names", adata_raw.var_names)

    # 2) handle duplicates
    if collapse_var:
        # collapse duplicated var names by summing columns (good for counts)
        def _collapse(A, names):
            df = pd.DataFrame.sparse.from_spmatrix(A) if sparse.issparse(A) else pd.DataFrame(A)
            df.columns = names
            return df.groupby(level=0, axis=1).sum()
        # collapse both objects on var (genes)
        A1 = _collapse(adata.X, adata.var_names)
        A2 = _collapse(adata_raw.X, adata_raw.var_names)
        # keep obs in original order
        A1.index = adata.obs_names
        A2.index = adata_raw.obs_names
        # replace X and var_names with collapsed
        adata = adata[:, []].copy()
        adata.X = A1.values
        adata.var_names = A1.columns.astype(str)
        adata.obs_names = A1.index.astype(str)

        adata_raw = adata_raw[:, []].copy()
        adata_raw.X = A2.values
        adata_raw.var_names = A2.columns.astype(str)
        adata_raw.obs_names = A2.index.astype(str)
    else:
        # just force uniqueness by appending suffixes
        adata.var_names_make_unique()
        adata.obs_names_make_unique()
        adata_raw.var_names_make_unique()
        adata_raw.obs_names_make_unique()

    # 3) intersect axes
    common_obs = adata.obs_names.intersection(adata_raw.obs_names)
    common_var = adata.var_names.intersection(adata_raw.var_names)

    ad_sub  = adata[common_obs, common_var].copy()
    raw_sub = adata_raw[common_obs, common_var]

    # 4) attach raw into a layer
    Xraw = raw_sub.X
    if sparse.issparse(Xraw):
        Xraw = Xraw.copy()
    else:
        Xraw = np.asarray(Xraw).copy()
    ad_sub.layers[layer_name] = Xraw

    print(f"[attach_counts] kept {ad_sub.n_obs} cells and {ad_sub.n_vars} genes "
          f"(intersected). Raw stored in .layers['{layer_name}'].")
    return ad_sub

In [16]:
# If you want to simply make names unique (fastest):
adata = attach_counts_intersection_safe(adata, adata_raw, layer_name="counts", collapse_var=False)

# If your var (gene) names are symbols with duplicates and you prefer to SUM duplicates first:
# adata = attach_counts_intersection_safe(adata, adata_raw, layer_name="counts", collapse_var=True)

[WARN] adata.obs_names: 96 duplicated names e.g. ['aggekfip-1', 'lapokbab-1', 'dilbcmig-1', 'aabdhpop-1', 'ajnddggj-1']
[OK] adata.var_names: all unique
[WARN] adata_raw.obs_names: 98 duplicated names e.g. ['aggekfip-1', 'lapokbab-1', 'dilbcmig-1', 'aabdhpop-1', 'ajnddggj-1']
[OK] adata_raw.var_names: all unique
[attach_counts] kept 980474 cells and 5101 genes (intersected). Raw stored in .layers['counts'].


In [20]:
adata.layers['counts']

<980474x5101 sparse matrix of type '<class 'numpy.float32'>'
	with 545822587 stored elements in Compressed Sparse Row format>

In [121]:
adata.write('../data/mtDNA_DSB_5k_clustered_manual_annotation_with_raw.h5ad')

In [105]:
list(adata.obs.cell_class.unique())

['Mature oligodendrocytes',
 'Excitatory neurons',
 'Telencephalon astrocytes',
 'Microglia',
 'Vascular leptomeningeal cells',
 'Inhibitory neurons',
 'Mature oligodendrocytes (mic)',
 'Excitatory neurons (cortex)',
 'Oligodendrocytes precursor cells',
 'Pericytes',
 'Vascular endothelial cells',
 'Endothelial cells',
 'Neural progenitors',
 'Ependymal cells',
 'Cholinergic neurons',
 'Chorid plexus epithelial cells',
 'Unknown',
 'Neurons',
 'Olfactory astrocytes',
 'Striatal neurons',
 'Excitatory neurons (thalamus)',
 'D1 medium spiny neurons (striatum)']

In [106]:
adata_OL = adata[adata.obs.cell_class.isin(['Mature oligodendrocytes','Oligodendrocytes precursor cells'])]

In [107]:
adata_sub = adata_OL[adata_OL.obs["condition"]=='mtDSB']
adata_sub = adata_sub[adata_sub.obs["age"]=='60']

adata_rest = adata_OL[adata_OL.obs["condition"]!='mtDSB']
adata_rest = adata_rest[adata_rest.obs["age"]=='60']

In [112]:
import numpy as np
import pandas as pd

# mean expression per gene for each group
mean_interest = np.asarray(adata_sub.layers['counts'].mean(axis=0)).ravel()
mean_rest = np.asarray(adata_rest.layers['counts'].mean(axis=0)).ravel()

diff = mean_interest - mean_rest
fold = (mean_interest + 1e-6) / (mean_rest + 1e-6)

# rough z-score of differences (per gene relative to all genes)
zscore = (diff - diff.mean()) / diff.std()

diff_df = pd.DataFrame({
    "gene": adata.var_names,
    "mean_interest": mean_interest,
    "mean_rest": mean_rest,
    "diff": diff,
    "log2FC": np.log2(fold),
    "zscore": zscore
}).sort_values("zscore", ascending=False)

diff_df.head()

,gene,mean_interest,mean_rest,diff,log2FC,zscore
559,Car2,9.893600,8.733556,1.160045,0.179926,20.428122
3736,Ptgds,12.537236,11.568737,0.968499,0.115988,17.031055
3148,Nmu,1.091608,0.165682,0.925926,2.719962,16.276026
2502,Kif5a,3.004230,2.324126,0.680104,0.370307,11.916360
2953,Mt2,1.599930,0.971784,0.628146,0.719300,10.994879


In [114]:
import numpy as np
import pandas as pd

# mean expression per gene for each group
mean_interest = np.asarray(adata_sub.layers['counts'].mean(axis=0)).ravel()
mean_rest = np.asarray(adata_rest.layers['counts'].mean(axis=0)).ravel()

diff = mean_interest - mean_rest
fold = (mean_interest + 1e-6) / (mean_rest + 1e-6)

# rough z-score of differences (per gene relative to all genes)
zscore = (diff - diff.mean()) / diff.std()

diff_df = pd.DataFrame({
    "gene": adata.var_names,
    "mean_interest": mean_interest,
    "mean_rest": mean_rest,
    "diff": diff,
    "log2FC": np.log2(fold),
    "zscore": zscore
}).sort_values("zscore", ascending=False)

In [115]:
diff_df_sorted = diff_df.sort_values(by=[ "mean_interest",'log2FC'], ascending=[False, False])
print(diff_df_sorted[diff_df_sorted.log2FC > 0.3].head(60))

          gene  mean_interest  mean_rest      diff    log2FC     zscore
2502     Kif5a       3.004230   2.324126  0.680104  0.370307  11.916360
3056     Ndrg2       1.994455   1.583211  0.411243  0.333140   7.148103
2953       Mt2       1.599930   0.971784  0.628146  0.719300  10.994879
2503     Kif5b       1.555145   1.213849  0.341296  0.357460   5.907590
1536   Fam107a       1.373556   1.026390  0.347165  0.420336   6.011676
1988     Gstp1       1.200508   0.789305  0.411203  0.604990   7.147393
2421       Jun       1.094199   0.880604  0.213595  0.313310   3.642805
3148       Nmu       1.091608   0.165682  0.925926  2.719962  16.276026
2182     Hsph1       1.040490   0.771697  0.268793  0.431156   4.621738
380        B2m       1.008218   0.735459  0.272759  0.455090   4.692076
1814      Gfap       0.948058   0.417827  0.530232  1.182070   9.258366
2176     Hspa5       0.902075   0.677179  0.224896  0.413709   3.843216
330       Atf4       0.821494   0.640203  0.181292  0.359722   3

In [111]:
diff_df_sorted = diff_df.sort_values(by=['log2FC',"mean_interest"], ascending=[False, False])
print(diff_df_sorted[diff_df_sorted.log2FC > 0.2].head(60))

           gene  mean_interest  mean_rest      diff    log2FC
4268    Slc5a12       0.000132   0.000000  0.000132  7.050923
4778      Trib3       0.037261   0.001505  0.035757  4.629125
690        Cd40       0.031358   0.001593  0.029765  4.297874
1804      Gdf15       0.009343   0.000509  0.008835  4.195622
1056       Cst7       0.008422   0.000708  0.007714  3.570251
2265    Il13ra2       0.000263   0.000022  0.000241  3.513824
764      Cdkn1a       0.057435   0.006329  0.051106  3.181682
2135      Hoxb5       0.001410   0.000155  0.001255  3.177964
3307        Oxt       0.001617   0.000221  0.001395  2.863486
4034      Scimp       0.000320   0.000044  0.000275  2.824498
617        Ccl3       0.005396   0.000775  0.004621  2.798792
3148        Nmu       1.091608   0.165682  0.925926  2.719962
359    Atp6v0d2       0.000658   0.000111  0.000547  2.561328
4118   Serpine1       0.003234   0.000553  0.002680  2.545013
663        Cd22       0.000639   0.000111  0.000529  2.519572
1140    

In [120]:
genes = ["Hif1a","Nfe2l2","Akt1","Pik3cd","Hmox1","Txnip","Slc16a1","Slc16a3",
         "Ldha","Ldhb",'Aldoa',"Mfn1","Mfn2","Opa1","Pkm","Sirt2","Ppargc1a"]
print(diff_df_sorted[diff_df_sorted.gene.isin(genes)])

          gene  mean_interest  mean_rest      diff    log2FC    zscore
180      Aldoa       3.061704   2.607455  0.454249  0.231692  7.910807
2608      Ldhb       1.276877   1.164345  0.112532  0.133100  1.850438
4189     Sirt2       0.875800   0.821292  0.054508  0.092706  0.821379
2607      Ldha       0.455825   0.393158  0.062668  0.213372  0.966097
3498       Pkm       0.412382   0.359498  0.052884  0.197996  0.792578
2082     Hif1a       0.386795   0.369330  0.017465  0.066660  0.164433
2862      Mfn1       0.267019   0.237040  0.029978  0.171808  0.386354
2863      Mfn2       0.220387   0.201170  0.019216  0.131617  0.195484
168       Akt1       0.215611   0.189199  0.026412  0.188526  0.323103
3281      Opa1       0.205777   0.192120  0.013657  0.099075  0.096895
4213   Slc16a1       0.167607   0.161804  0.005802  0.050830 -0.042409
3099    Nfe2l2       0.112290   0.089998  0.022292  0.319260  0.250031
4852     Txnip       0.093172   0.083337  0.009835  0.160929  0.029100
3603  

## Functional Modules in mtDSB Oligodendrocytes

### 1. Oxidative Stress & Detoxification
- **Mt2** – metallothionein, binds Zn/Cu, scavenges ROS  
- **Gstp1** – glutathione detox enzyme  
- **Sqstm1 (p62)** – oxidative stress sensor, links ROS to autophagy  
- **Nfe2l1** – TF regulating antioxidant genes  
- **Hspd1, Hspa9** – mitochondrial chaperones for ROS stress  
➡️ Evidence for **oxidative stress and mitochondrial redox imbalance**

---

### 2. Integrated Stress Response (ISR) & UPRmt
- **Atf4, Jun** – stress-activated TFs driving ISR/UPRmt  
- **Hspa5 (BiP/GRP78), Hspd1 (HSP60), Hspa9** – ER/mitochondrial chaperones  
- **Hsph1** – HSP110 family, protein folding/stress tolerance  
➡️ Indicates **protein misfolding and translational stress downstream of mitochondrial dysfunction**

---

### 3. Antigen Presentation & Immune Signaling
- **B2m, H2-D1, H2-K1** – MHC-I antigen presentation  
- **Ctss (Cathepsin S)** – lysosomal protease for antigen processing  
- **Cd40, Cd44, Cd22, Cd3e** – co-stimulatory/immune interaction molecules  
➡️ Suggests **stressed OLs present antigen and engage immune surveillance**

---

### 4. Axonal/Myelin Transport Stress
- **Kif5a, Kif5b** – kinesin motors for axonal/myelin cargo  
- **Dync1li1** – dynein light intermediate chain, retrograde transport  
- **Ptpra, Itgb1, Mpzl1, Sorbs1** – adhesion/cell interaction molecules  
➡️ Points to **disturbed axonal transport and OL–axon coupling**

---

### 5. Astrocytic Reactivity & Glial Crosstalk
- **Ndrg2, Gfap, Mlc1** – hallmark astrocytic/reactive gliosis genes  
- **S100a1, Calb2** – calcium-binding proteins in reactive astrocytes  
➡️ Reflects **astrocyte activation secondary to OL mtDSB stress**

---

### 6. Inflammatory Signaling & Cytokines
- **Nmu (Neuromedin U)** – neuropeptide with pro-inflammatory activity  
- **Ccl3 (MIP-1α)** – chemokine recruiting myeloid cells  
- **Irf4** – immune TF controlling cytokine expression  
➡️ Suggests **immune-modulatory signaling in the OL/astro niche**

---

### 7. Metabolic Remodeling
- **Hadhb** – mitochondrial β-oxidation enzyme  
- **Parvb, Sorbs1** – cytoskeletal/metabolic adaptors  
- **Fosl1, Myc** – TFs linked to metabolic reprogramming and proliferation  
➡️ Evidence for **shifts in mitochondrial and metabolic regulation**

---

## ✅ Overall Takeaway
mtDSB oligodendrocytes show a **coherent multi-pathway stress program**:
- **Redox stress** → metallothioneins, glutathione enzymes  
- **Mitochondrial/ER stress** → ISR/UPRmt activation  
- **Immune presentation** → MHC-I and lysosomal proteases  
- **Axonal transport disruption** → kinesins/dyneins  
- **Astro reactivity** → Ndrg2, Gfap upregulation  
- **Pro-inflammatory signaling** → Nmu, Ccl3  
➡️ Together, these changes suggest OLs under mtDNA damage are **alive but stressed**, tipping the microenvironment toward **immune activation and glial crosstalk**.